In [2]:
import numpy as np
import pandas as pd
import pickle
from datasets import load_dataset, Dataset, load_from_disk
import ast
import torch
import torch.nn as nn
#import bitsandbytes as bnb
from peft import LoraConfig, get_peft_model, PeftModel, PeftConfig
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM, TrainingArguments, DataCollatorForLanguageModeling, Trainer
from datasets import load_dataset, Dataset, DatasetDict
from dataclasses import dataclass, field
from typing import Optional
from trl import SFTTrainer
import random
from tqdm.notebook import tqdm  # Use notebook-specific version


In [3]:

def cut_max_len(tokenizer, note, answer, model_name):
    if 'instruct' in model_name :  # for instruct llms
        model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
        instruct_template = \
        """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>

{user_msg}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

        """

        system_prompt = """As a medical expert, your task is to carefully analyze the clinical note and complete the following steps:
1. Identify all key medical terms within the clinical note. These terms should include: Diagnoses, Symptoms and Relevant conditions
2. For each medical term identified, assign the MOST APPROPRIATE ICD code, ensuring accuracy and specificity in your choices.
3. The clinical note may contain multiple conditions, and your role is to ensure that each one is identified and accurately mapped to the correct ICD code.
Your response should follow this structured output format: [medical term] corresponds to [ICD code].
If no ICD code is found for a term: [term] does not have a matching ICD code."""
        user_msg = "Clinical Note: {}"
        prompt = instruct_template.format(system_prompt=system_prompt,user_msg=user_msg )
        model_answer = "### Answer: {}<|eot_id|>"

    else:
        
        model_name = "meta-llama/Meta-Llama-3-8B"
        prompt = """As a medical expert, your task is to carefully analyze the clinical note and complete the following steps:
1. Identify all key medical terms within the clinical note. These terms should include: Diagnoses, Symptoms and Relevant conditions
2. For each medical term identified, assign the MOST APPROPRIATE ICD code, ensuring accuracy and specificity in your choices.
3. The clinical note may contain multiple conditions, and your role is to ensure that each one is identified and accurately mapped to the correct ICD code.
Your response should follow this structured output format: [medical term] corresponds to [ICD code].
If no ICD code is found for a term: [term] does not have a matching ICD code.
### Clinical Note: {note}"""
        model_answer = "### Output: {answer}"

    
    prompt = prompt.format(note=note)
    
    tokenized_prompt = tokenizer(prompt)

    tokenized_answer = tokenizer(model_answer.format(answer=answer))

    max_token_length = 2048 - len(tokenized_answer['input_ids'])

    # If the prompt is too long, cut it down
    if len(tokenized_prompt['input_ids']) > max_token_length:
        tokenized_prompt['input_ids'] = tokenized_prompt['input_ids'][:max_token_length]

    # Decode the shortened prompt back into text
    shortened_text = tokenizer.decode(tokenized_prompt['input_ids'], skip_special_tokens=True)
    
    return shortened_text        



In [4]:

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
with open(file='data/df_train.pkl', mode='rb') as f:
    train_dataset = pickle.load(f)
with open(file='data/df_test.pkl', mode='rb') as f:
    test_dataset = pickle.load(f)

In [ ]:
tqdm.pandas()
test_dataset['prompt'] = test_dataset.progress_apply(
    lambda row: cut_max_len(tokenizer=tokenizer, note=row['note'], answer=row['answer'], model_name=model_name), axis=1)

train_dataset['prompt'] = train_dataset.progress_apply(
    lambda row: cut_max_len(tokenizer=tokenizer, note=row['note'], answer=row['answer'], model_name=model_name), axis=1)

In [10]:
new_test_dataset = test_dataset.drop(columns=['note'])
new_test_dataset.to_pickle('df_test_prompt.pkl')


new_train_dataset = train_dataset.drop(columns=['note'])
new_train_dataset.to_pickle('df_train_prompt.pkl')